#### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import pickle
import warnings
import os
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings('ignore')

from kmrf import KMRF
from KMRF_training_config import *
from SIMULATOR import SIMULATOR


# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"  Pandas version: {pd.__version__}")
print(f"  NumPy version: {np.__version__}")

✓ Libraries imported successfully
  Pandas version: 2.3.3
  NumPy version: 2.3.4


## Multi-Horizon Regime Predictions

#### Global Vars and Helper Functions

In [2]:
KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')
KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')

# Using saved KAMA+MSR models
def get_KM_model_dates(KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')) -> pd.Series:
    return pd.Series([f.stem for f in list(KM_MODEL_BASE_PATH.glob('*'))]).sort_values().iloc[1:].reset_index(drop=True)

def get_KM_model_paths(MODEL_DATE: str, KM_MODEL_BASE_PATH = Path('saved_models/KAMA_MSR/us_equity')) ->  pd.Series:
    return pd.Series(list((KM_MODEL_BASE_PATH / MODEL_DATE).glob('*'))).sort_values().reset_index(drop=True)

# Using saved KMRF predictions
def get_asset_names(KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')) -> pd.Series:
    kmrf_preds_paths = list(KMRF_PREDICTIONS_BASE_PATH.glob('*'))
    return pd.Series([f.stem.split('multi')[0][:-1].replace('_', ' ') for f in kmrf_preds_paths]).sort_values().reset_index(drop=True)

def get_KMRF_prediction_paths(KMRF_PREDICTIONS_BASE_PATH = Path('data/multi_horizon_predictions')) -> pd.Series:
    return pd.Series(list(KMRF_PREDICTIONS_BASE_PATH.glob('*'))).sort_values().reset_index(drop=True)

#### --------------------------------------------------------

In [3]:
KM_MODEL_DATES = get_KM_model_dates()
ASSET_NAMES = get_asset_names()

display(ASSET_NAMES)

0     Consumer Discretionary Select Sector SPDR
1           Consumer Staples Select Sector SPDR
2                     Energy Select Sector SPDR
3                  Financial Select Sector SPDR
4                Health Care Select Sector SPDR
5                 Industrial Select Sector SPDR
6                        Invesco DB Agriculture
7                             Invesco QQQ Trust
8                  Materials Select Sector SPDR
9         SPDR Dow Jones Industrial Average ETF
10                             SPDR Gold Shares
11                             SPDR S&P 500 ETF
12                Technology Select Sector SPDR
13               United States Natural Gas Fund
14                       United States Oil Fund
15                 Utilities Select Sector SPDR
16          Vanguard FTSE Developed Markets ETF
17           Vanguard FTSE Emerging Markets ETF
18                     Vanguard FTSE Europe ETF
19                  iShares China Large-Cap ETF
20                       iShares MSCI In

In [16]:
ASSET_INDEX = 11
sim = SIMULATOR(km_model_path=get_KM_model_paths(MODEL_DATE=KM_MODEL_DATES[0])[ASSET_INDEX], kmrf_preds_path=get_KMRF_prediction_paths()[ASSET_INDEX])
sim.show_class_variables()

regime_to_int: <class 'dict'>
int_to_regime: <class 'dict'>
km_model_path: <class 'pathlib._local.PosixPath'>
kmrf_preds_path: <class 'pathlib._local.PosixPath'>
km_model: <class 'kama_msr.KAMA_MSR'>
km_model_date: <class 'str'>
asset_name: <class 'str'>
kmrf_predictions: <class 'pandas.core.frame.DataFrame'>
kmrf_predictions_int: <class 'pandas.core.frame.DataFrame'>
regime_pred_rankings: <class 'pandas.core.frame.DataFrame'>
regime_pred_rankings_int: <class 'pandas.core.frame.DataFrame'>


In [17]:
sim.kmrf_predictions

model_end_date  P(LV_Bull)  P(LV_Bear)  \
date       horizon prediction_date                                          
2019-01-02 1       2019-01-02            20181231    0.005484    0.738907   
           2       2019-01-03            20181231    0.002844    0.772020   
           3       2019-01-04            20181231    0.003930    0.787103   
           4       2019-01-07            20181231    0.003230    0.802660   
           5       2019-01-08            20181231    0.002842    0.782096   
...                                           ...         ...         ...   
2025-10-31 17      2025-11-25            20251007    0.992879    0.001189   
           18      2025-11-26            20251007    0.991069    0.001485   
           19      2025-11-28            20251007    0.990190    0.001576   
           20      2025-12-01            20251007    0.992680    0.001024   
           21      2025-12-02            20251007    0.990339    0.001413   

                                    P(HV_Bull)  P(HV_Bear)  
date       horizon prediction_date                          
2019-01-02 1       2019-01-02         0.015601    0.240008  
           2       2019-01-03         0.009267    0.215869  
           3       2019-01-04         0.008758    0.200209  
           4       2019-01-07         0.007942    0.186167  
           5       2019-01-08         0.005851    0.209211  
...                                        ...         ...  
2025-10-31 17      2025-11-25         0.003100    0.002832  
           18      2025-11-26         0.003692    0.003754  
           19      2025-11-28         0.003913    0.004321  
           20      2025-12-01         0.002428    0.003867  
           21      2025-12-02         0.002538    0.005709  

[36099 rows x 5 columns]

In [18]:
sim.regime_pred_rankings

regime_ranking
date       horizon prediction_date                                                  
2019-01-02 1       2019-01-02       [P(LV_Bear), P(HV_Bear), P(HV_Bull), P(LV_Bull)]
           2       2019-01-03       [P(LV_Bear), P(HV_Bear), P(HV_Bull), P(LV_Bull)]
           3       2019-01-04       [P(LV_Bear), P(HV_Bear), P(HV_Bull), P(LV_Bull)]
           4       2019-01-07       [P(LV_Bear), P(HV_Bear), P(HV_Bull), P(LV_Bull)]
           5       2019-01-08       [P(LV_Bear), P(HV_Bear), P(HV_Bull), P(LV_Bull)]
...                                                                              ...
2025-10-31 17      2025-11-25       [P(LV_Bull), P(HV_Bull), P(HV_Bear), P(LV_Bear)]
           18      2025-11-26       [P(LV_Bull), P(HV_Bear), P(HV_Bull), P(LV_Bear)]
           19      2025-11-28       [P(LV_Bull), P(HV_Bear), P(HV_Bull), P(LV_Bear)]
           20      2025-12-01       [P(LV_Bull), P(HV_Bear), P(HV_Bull), P(LV_Bear)]
           21      2025-12-02       [P(LV_Bull), P(HV_Bear), P(HV_Bull), P(LV_Bear)]

[36099 rows x 1 columns]